In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split

from tools import tokenize_df, make_encoders
from tools import tuning_sgd, tuning_sgd_partial, tuning_logistic_regression, tuning_linear_svc
from tools import cross_validation

version = 0

In [4]:
df = pd.read_csv("../datasets/porn_detection/train.csv")
df_test = pd.read_csv("../datasets/porn_detection/test.csv")
df = df.dropna(subset=['title'])
df_test = df_test.dropna(subset=['title'])

In [9]:
print(df.shape[0], f"0={sum(df["label"] == 0)}", f"1={sum(df["label"] == 1)}")
print(df.shape[0], f"0={sum(df["label"] == 0)/df.shape[0]}", f"1={sum(df["label"] == 1)/df.shape[0]}")

135308 0=118593 1=16715
135308 0=0.8764670233836875 1=0.1235329766163124


In [ ]:
tokenize_df(df)
tokenize_df(df_test)

In [ ]:
encoder, encoder_wb, encoder_url = make_encoders()
encoder_s, encoder_wb_s, encoder_url_s = make_encoders()

y = df["label"]
id_train, id_test,  = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y)
X_train = hstack([encoder.fit_transform(df["lemmatized"].iloc[id_train]),
                  encoder_wb.fit_transform(df["lemmatized"].iloc[id_train]),
                  encoder_url.fit_transform(df["url"].iloc[id_train])]).tocsr()
X_test = hstack([encoder.transform(df["lemmatized"].iloc[id_test]), 
                 encoder_wb.transform(df["lemmatized"].iloc[id_test]),
                 encoder_url.transform(df["url"].iloc[id_test])]).tocsr()

X_train_subm = hstack([encoder_s.fit_transform(df["lemmatized"]), 
                       encoder_wb_s.fit_transform(df["lemmatized"]),
                       encoder_url_s.fit_transform(df["url"])]).tocsr()
X_test_subm = hstack([encoder_s.transform(df_test["lemmatized"]), 
                      encoder_wb_s.transform(df_test["lemmatized"]),
                      encoder_url_s.transform(df_test["url"])]).tocsr()

y_train = y.iloc[id_train]
y_test = y.iloc[id_test]
y_train_subm = y
print(f"Размеры ТРЕНИРОВОЧНОЙ выборки {X_train.shape}")
print(f"Размеры ТЕСТОВОЙ выборки {X_test.shape}")
print(f"Размеры выборки для submission {X_test_subm.shape}")

Размеры ТРЕНИРОВОЧНОЙ выборки (108246, 417947)
Размеры ТЕСТОВОЙ выборки (27062, 417947)
Размеры выборки для submission (165378, 479988)


### Создание файлов с логами тюнинга гиперпараметров

In [ ]:
tuning_sgd(X_train, y_train, X_test, y_test)
tuning_sgd_partial(X_train, y_train, X_test, y_test)
tuning_logistic_regression(X_train, y_train, X_test, y_test)
tuning_linear_svc(X_train, y_train, X_test, y_test)

### Validation

In [ ]:
cross_validation(X_train, y_train,
                X_test, y_test, 
                X_train_subm, X_test_subm, y_train_subm,
                df_test)

######################################################################
MODEL TYPE: SGDClassifier_fit
best CV mean_f1 = 0.9833, mean_recall = 0.9794
best params: {'alpha': 1e-06, 'class_weight': {0: 0.570472416046546, 1: 4.0474872868680825}, 'eta0': 0.001, 'learning_rate': 'optimal', 'loss': 'log_loss', 'max_iter': 1000, 'n_iter_no_change': 10, 'penalty': 'l2', 'tol': 0.0001, 'validation_fraction': 0.1}
----------------------------------------------------------------------
              precision    recall  f1-score   support

           0     0.9974    0.9981    0.9977     23719
           1     0.9865    0.9815    0.9840      3343

    accuracy                         0.9960     27062
   macro avg     0.9919    0.9898    0.9909     27062
weighted avg     0.9960    0.9960    0.9960     27062

Confusion matrix:
[[23674    45]
 [   62  3281]]
submission saved: submission_SGDClassifier_fit.csv
######################################################################

########################

# Final LB score
SGDClassifier_fit - 0.98510  
SGDClassifier_part_fit - 0.98527  
LogisticRegression - 0.98590  
LinearSVC - 0.98607  